In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal as _lal

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

%reload_ext autoreload
%autoreload 2

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())
print("sys.path:", sys.path[:5])

PROJECT_ROOT: /home/victor/gw/cbc_pe
cwd: /home/victor/gw/cbc_pe/notebooks
sys.path: ['/home/victor/gw/cbc_pe', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '']


In [ ]:
import numpy as np
from dataclasses import asdict
from src.config import SimulationConfig
from src.parameters import CBCParameters
from src.dataset import DatasetBuilder

rng = np.random.default_rng(1234)

config = SimulationConfig(
    sampling_frequency=4096.0, # Hz
    duration=4.0, # seconds
    low_frequency_cutoff=30.0, # Hz
    waveform_approximant="SEOBNRv4_opt",
    target_network_snr_range=(10.0, 25.0),
)

builder = DatasetBuilder.from_config(
    config=config,
    detector_names=["H1", "L1", "V1"],
    signal_processor_kwargs={
        "apply_whitening": True,
        "apply_lowpass": True,
        "apply_highpass": True,
        "apply_standardization": True,
    },
    rng=rng,
)

samples = []

for _ in range(100):
    params = CBCParameters(
        mass_1=20.0,
        mass_2=20.0,
        distance=rng.uniform(200.0, 5000.0),
        inclination=np.arccos(rng.uniform(-1.0, 1.0)),
        ra=rng.uniform(0.0, 2.0*np.pi),
        dec=np.arcsin(rng.uniform(-1.0, 1.0)),
        spin_1z=rng.uniform(-1.0, 1.0),
        spin_2z=rng.uniform(-1.0, 1.0),
        polarization_angle=rng.uniform(0.0, 2.0*np.pi),
    )

    samples.append(builder.build_sample(params=params))

X = np.stack([s.X for s in samples])
y = np.stack([s.y for s in samples])


KeyboardInterrupt: 

In [17]:
params_dict = {
    "mass_1": np.array([s.parameters.mass_1 for s in samples]),
    "mass_2": np.array([s.parameters.mass_2 for s in samples]),
    "distance": np.array([s.parameters.distance for s in samples]),
    "inclination": np.array([s.parameters.inclination for s in samples]),
    "ra": np.array([s.parameters.ra for s in samples]),
    "dec": np.array([s.parameters.dec for s in samples]),
    "spin_1z": np.array([s.parameters.spin_1z for s in samples]),
    "spin_2z": np.array([s.parameters.spin_2z for s in samples]),
    "polarization_angle": np.array([s.parameters.polarization_angle for s in samples]),
    "total_mass": np.array([s.parameters.total_mass for s in samples]),
    "chirp_mass": np.array([s.parameters.chirp_mass for s in samples]),
    "chi_eff": np.array([s.parameters.chi_eff for s in samples]),
}

In [ ]:
np.savez(
    "cbc_100_m1_20_m2_20.npz",
    X=X,
    network_snrs=network_snrs,
    injection_times=injection_times,
    detector_names=np.array(["H1", "L1", "V1"]),
    **{f"params_{k}": v for k, v in params_dict.items()},
)

In [ ]:
data = np.load("cbc_100_m1_20_m2_20.npz")

print(data.files)

params = {
    k.replace("params_", ""): data[k]
    for k in data.files
    if k.startswith("params_")
}




['X', 'y', 'network_snrs', 'injection_times', 'detector_names', 'params_mass_1', 'params_mass_2', 'params_distance', 'params_inclination', 'params_ra', 'params_dec', 'params_spin_1z', 'params_spin_2z', 'params_polarization_angle', 'params_total_mass', 'params_chirp_mass', 'params_chi_eff']
